In [ ]:
%load_ext autoreload
%autoreload 2
%time

In [ ]:
import sys, os 
sys.path.append("../../")
from data_loading import load_runs

RUN = ["1"]  # this can be a list of several runs, i.e. [1,2,3]
blinded = True

rundata, mc_weights, data_pot = load_runs(
    RUN,
    data="bnb",  # which data to load
    # truth_filtered_sets=["nue", "drt", "nc_pi0", "cc_pi0", "cc_nopi", "cc_cpi", "nc_nopi", "nc_cpi"],
    # Which truth-filtered MC sets to load in addition to the main MC set. At least nu_e and dirt
    # are highly recommended because the statistics at the final level of the selection are very low.
    truth_filtered_sets=["nue", "drt"],
    # Choose which additional variables to load. Which ones are required may depend on the selection
    # you wish to apply.
    loadpi0variables=True,
    loadshowervariables=True,
    loadrecoveryvars=True,
    loadsystematics=True,
    loadnumuvariables=True, # True for 1mu1p/ False for 1e1p
    numupresel=False,
    # Load the nu_e set one more time with the LEE weights applied
    load_lee=False,
    # With the cache enabled, by default the loaded dataframes will be stored as HDF5 files
    # in the 'cached_dataframes' folder. This will speed up subsequent loading of the same data.
    enable_cache=True,
    # Since this is Open Data, we are allowed to unblind the data. By default, the data is blinded.
    blinded=blinded,
    load_numu_tki=True,
    load_nue_tki=True
)

In [ ]:
from microfit import selections as sel
import pandas as pd
import numpy as np

# Choose your selection and preselection category
# selection = "SG_1MU1P"
# preselection = "SG_1MUNP"
selection   =  "TKI_1mu1p" #"SIGNAL_1MU1P"#"TKI_1mu1p" #"TKI_1mu1p" #"SG_1MU1P" 
preselection = "SG_1MUNP" #"SIGNAL_1MUNP"#"SG_1MUNP"NUMU

# Build the full query (usually e.g. sel_CCNp0pi and sel_CC1p0pi)
query = f"{sel.preselection_categories[preselection]['query']} and {sel.selection_categories[selection]['query']}"
sel_title = sel.selection_categories[selection]['title']

# All available MC + EXT (excluding real data)
#all_mc = pd.concat([df for k, df in rundata.items() if k != 'data'])
all_mc  = pd.concat([df for k, df in rundata.items() if k not in ['data', 'ext']])


# Define the signal category
all_sig = all_mc['category_1mu1p'] == 23
#all_sig = all_mc['Signal_1mu1p'] == True

# Total signal and background before cuts
tot_sig = np.sum(all_mc.loc[all_sig, 'weights'])
tot_bkg = np.sum(all_mc.loc[~all_sig, 'weights'])
print(f"Total candidate signal events (before selection): {tot_sig}")
print(f"Total candidate background events (before selection): {tot_bkg}")

# Apply selection
sel_mc = all_mc.query(query, engine='python')

# Signal and background among selected events
is_sig = sel_mc['category_1mu1p'] == 23  # Gardiner's style
#is_sig = sel_mc['Signal_1mu1p'] == True   # Truth-Level study
sel_sig = np.sum(sel_mc.loc[is_sig, 'weights'])
sel_bkg = np.sum(sel_mc.loc[~is_sig, 'weights'])
sel_evt = np.sum(sel_mc['weights'])

print('\nAfter cuts:')
print(f"Total signal events (selected): {sel_sig}")
print(f"Total background events (selected): {sel_bkg}")
print(f"Total selected events (sig + bkg): {sel_evt}")
print(f"Check: sel_sig + sel_bkg = {sel_sig + sel_bkg}\n")

# Compute efficiency and purity
efficiency = (sel_sig / tot_sig) * 100 if tot_sig > 0 else 0
purity = (sel_sig / sel_evt) * 100 if sel_evt > 0 else 0

print(f"Selection: {sel_title}")
print(f"Efficiency: {efficiency:.2f}%")
print(f"Purity: {purity:.2f}%")


## ECAL_vs_pT_phiT

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

slices_pn_1mu1p = [
    '0 < RecoPN_1mu1p < 0.1',
    '0.1 < RecoPN_1mu1p < 0.2',
    '0.2 < RecoPN_1mu1p < 0.3',
    '0.3 < RecoPN_1mu1p < 0.4',
    '0.4 < RecoPN_1mu1p < 0.5',
    '0.5 < RecoPN_1mu1p < 0.6',
    '0.6 < RecoPN_1mu1p < 0.7',
    '0.7 < RecoPN_1mu1p < 0.8',
    '0.8 < RecoPN_1mu1p < 1.2',
    '0.0 < RecoPN_1mu1p < 0.2',
]
print("Min and Max of RecoPN_1mu1p:", sel_mc['RecoPN_1mu1p'].min(), sel_mc['RecoPN_1mu1p'].max())


# Binding energy (GeV)
binding_energy = 0.02478 #0.03
proton_mass    = 0.9382720813


# Setup subplot grid
num_slices = len(slices_pn_1mu1p)
ncols = 3  # number of columns per row
nrows = int(np.ceil(num_slices / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), sharex=True, sharey=False)
axes = axes.flatten()  # Flatten in case of 2D layout

# Stats storage
ecal_rms_list = []
ecal_mean_list = []
ecal_std_list = []
slice_labels = []

for idx, s in enumerate(slices_pn_1mu1p):
    df = sel_mc.query(s, engine='python')

    reco_muon_energy = df["RecoMuonE_1muNp"]
    reco_proton_energy = df["RecoLeadProtonE_1muNp"]
    true_muon_energy = df["TrueMuonE_1muNp"]
    true_proton_energy = df["TrueLeadProtonE_1muNp"]
    
    reco_ecal = reco_muon_energy + (reco_proton_energy - proton_mass) + binding_energy
    true_ecal = df["nu_e"]
    resolution = (reco_ecal - true_ecal) / true_ecal

    # Stats
    mean = np.mean(resolution)
    std = np.std(resolution)
    rms = np.sqrt(np.mean(resolution ** 2))

    ecal_mean_list.append(mean)
    ecal_std_list.append(std)
    ecal_rms_list.append(rms)
    slice_labels.append(s.split('<')[-1].strip())

    # Plot in current subplot
    ax = axes[idx]
    ax.hist(resolution, bins=20, range=(-1.0, 1.0), histtype='stepfilled', color='navy', alpha=0.7)
    ax.set_title(f"{s}", fontsize=10)
    ax.text(0.95, 0.95, f"Mean: {mean:.3f}\nStd: {std:.3f}\nRMS: {rms:.3f}",
            transform=ax.transAxes, ha='right', va='top',
            bbox=dict(facecolor='white', alpha=0.8), fontsize=8)

    if idx % ncols == 0:
        ax.set_ylabel("Events")
    if idx >= num_slices - ncols:
        ax.set_xlabel("$E_\\mathrm{cal}$ resolution")

# Hide empty subplots (if any)
for j in range(len(slices_pn_1mu1p), len(axes)):
    fig.delaxes(axes[j])

# # Supertitle and layout
# fig.suptitle(f"Ecal Resolution in phiT Slices: {sel.selection_categories[selection]['title']}", fontsize=14)
# fig.tight_layout(rect=[0, 0, 1, 0.95])

# # Save single multi-panel figure
# plt.savefig(f'analysis_1mu1p/analysis_plots/resolution/ecal_resolution_grid_phiT_{preselection}_{selection}.pdf')
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

slices_pT_1mu1p = [
    '0 < RecoDeltaPT_1mu1p < 0.1',
    '0.1 < RecoDeltaPT_1mu1p < 0.2',
    '0.2 < RecoDeltaPT_1mu1p < 0.3',
    '0.3 < RecoDeltaPT_1mu1p < 0.4',
    '0.4 < RecoDeltaPT_1mu1p < 0.5',
    '0.5 < RecoDeltaPT_1mu1p < 0.6',
    '0.6 < RecoDeltaPT_1mu1p < 0.7',
    '0.7 < RecoDeltaPT_1mu1p < 0.8',
    '0.8 < RecoDeltaPT_1mu1p < 1.2',
    '0.0 < RecoDeltaPT_1mu1p < 0.2',
]
print("Min and Max of RecoPN_1mu1p:", sel_mc['RecoDeltaPT_1mu1p'].min(), sel_mc['RecoDeltaPT_1mu1p'].max())


# Setup subplot grid
num_slices = len(slices_pT_1mu1p)
ncols = 3  # number of columns per row
nrows = int(np.ceil(num_slices / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), sharex=True, sharey=False)
axes = axes.flatten()  # Flatten in case of 2D layout

# Stats storage
ecal_rms_list = []
ecal_mean_list = []
ecal_std_list = []
slice_labels = []

for idx, s in enumerate(slices_pT_1mu1p):
    df = sel_mc.query(s, engine='python')

    reco_muon_energy = df["RecoMuonE_1muNp"]
    reco_proton_energy = df["RecoLeadProtonE_1muNp"]
    true_muon_energy = df["TrueMuonE_1muNp"]
    true_proton_energy = df["TrueLeadProtonE_1muNp"]
    
    reco_ecal = reco_muon_energy + (reco_proton_energy - proton_mass) + binding_energy
    true_ecal = df["nu_e"]
    resolution = (reco_ecal - true_ecal) / true_ecal

    # Stats
    mean = np.mean(resolution)
    std = np.std(resolution)
    rms = np.sqrt(np.mean(resolution ** 2))

    ecal_mean_list.append(mean)
    ecal_std_list.append(std)
    ecal_rms_list.append(rms)
    slice_labels.append(s.split('<')[-1].strip())

    # Plot in current subplot
    ax = axes[idx]
    ax.hist(resolution, bins=20, range=(-1.0, 1.0), histtype='stepfilled', color='navy', alpha=0.7)
    ax.set_title(f"{s}", fontsize=10)
    ax.text(0.95, 0.95, f"Mean: {mean:.3f}\nStd: {std:.3f}\nRMS: {rms:.3f}",
            transform=ax.transAxes, ha='right', va='top',
            bbox=dict(facecolor='white', alpha=0.8), fontsize=8)

    if idx % ncols == 0:
        ax.set_ylabel("Events")
    if idx >= num_slices - ncols:
        ax.set_xlabel("$E_\\mathrm{cal}$ resolution")

# Hide empty subplots (if any)
for j in range(len(slices_pT_1mu1p), len(axes)):
    fig.delaxes(axes[j])

# # Supertitle and layout
# fig.suptitle(f"Ecal Resolution in phiT Slices: {sel.selection_categories[selection]['title']}", fontsize=14)
# fig.tight_layout(rect=[0, 0, 1, 0.95])

# # Save single multi-panel figure
# plt.savefig(f'analysis_1mu1p/analysis_plots/resolution/ecal_resolution_grid_phiT_{preselection}_{selection}.pdf')
plt.show()


## Efficiency/Purity in Each Slices

In [ ]:
# Per-slice purity & efficiency 

slice_queries = slices_pT_1mu1p  # or make a copy with overlaps removed

per_slice = []  

for s in slice_queries:
    # Before cuts: denominator for efficiency (signal only)
    before_df = all_mc.query(s, engine='python')
    denom_sig = np.sum(before_df.loc[before_df['category_1mu1p'] == 23, 'weights'])

    # After cuts: for purity + numerator for efficiency
    after_df = sel_mc.query(s, engine='python')
    sel_sig_slice = np.sum(after_df.loc[after_df['category_1mu1p'] == 23, 'weights'])
    sel_evt_slice = np.sum(after_df['weights'])

    eff_slice = (sel_sig_slice / denom_sig) * 100 if denom_sig > 0 else 0.0
    pur_slice = (sel_sig_slice / sel_evt_slice) * 100 if sel_evt_slice > 0 else 0.0

    per_slice.append({
        'slice': s,
        'sig_before': denom_sig,
        'sig_after': sel_sig_slice,
        'evt_after': sel_evt_slice,
        'efficiency_%': eff_slice,
        'purity_%': pur_slice,
    })

print(f"\nPer-slice metrics for {sel.selection_categories[selection]['title']} DeltapT slices):")
for r in per_slice:
    print(f"- {r['slice']}: eff={r['efficiency_%']:.2f}%  |  purity={r['purity_%']:.2f}%  "
          f"(sig_before={r['sig_before']:.2f}, sig_after={r['sig_after']:.2f}, evt_after={r['evt_after']:.2f})")


In [ ]:
# Per-slice purity & efficiency 

slice_queries = slices_pn_1mu1p  

per_slice = []  # hold dicts of results

for s in slice_queries:
    # Before cuts: denominator for efficiency (signal only)
    before_df = all_mc.query(s, engine='python')
    denom_sig = np.sum(before_df.loc[before_df['category_1mu1p'] == 23, 'weights'])

    # After cuts: for purity + numerator for efficiency
    after_df = sel_mc.query(s, engine='python')
    sel_sig_slice = np.sum(after_df.loc[after_df['category_1mu1p'] == 23, 'weights'])
    sel_evt_slice = np.sum(after_df['weights'])

    eff_slice = (sel_sig_slice / denom_sig) * 100 if denom_sig > 0 else 0.0
    pur_slice = (sel_sig_slice / sel_evt_slice) * 100 if sel_evt_slice > 0 else 0.0

    per_slice.append({
        'slice': s,
        'sig_before': denom_sig,
        'sig_after': sel_sig_slice,
        'evt_after': sel_evt_slice,
        'efficiency_%': eff_slice,
        'purity_%': pur_slice,
    })

print(f"\nPer-slice metrics for {sel.selection_categories[selection]['title']} (pN slices):")
for r in per_slice:
    print(f"- {r['slice']}: eff={r['efficiency_%']:.2f}%  |  purity={r['purity_%']:.2f}%  "
          f"(sig_before={r['sig_before']:.2f}, sig_after={r['sig_after']:.2f}, evt_after={r['evt_after']:.2f})")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


proton_mass = 0.938  # GeV
binding_energy = 0.03  # GeV

def compute_resolution_stats(df, label):
    reco_muon_energy = df["RecoMuonE_1muNp"]
    reco_proton_energy = df["RecoLeadProtonE_1muNp"]
    true_nu_energy = df["nu_e"]

    reco_ecal = reco_muon_energy + (reco_proton_energy - proton_mass) + binding_energy
    resolution = (reco_ecal - true_nu_energy) / true_nu_energy

    mean = resolution.mean()
    std = resolution.std()
    rms = np.sqrt((resolution ** 2).mean())
    stat_unc = std / np.sqrt(len(resolution))  # error on the mean

    print(f"{label}")
    print(f"  Events: {len(resolution)}")
    print(f"  Mean: {mean:.4f}, Std: {std:.4f}, RMS: {rms:.4f}, Stat. Unc.: {stat_unc:.4f}")
    return resolution, mean, std, rms, stat_unc

# All events 
res_all, mean_all, std_all, rms_all, staterr_all = compute_resolution_stats(sel_mc, r"All 1mu1p events")


# Selected slice 
#sel_sig['RecoAlpha3D_deg'] = np.rad2deg(sel_sig['RecoAlpha3D_1mu1p'])
sel_slice = sel_mc[
    (sel_mc["RecoDeltaPT_1mu1p"].values < 0.3)
    #& (sel_mc_1mu1p["RecoPN_1mu1p"].values < 0.22) 
     #& (sel_mc["RecoDeltaPhiT_1mu1p"].values < 1.0)   
       & (sel_mc["RecoPN_1mu1p"].values < 0.2)
]

res_cut, mean_cut, std_cut, rms_cut, staterr_cut = compute_resolution_stats(sel_slice, "pT < 0.2")


plt.figure(figsize=(9, 6))
bins = np.linspace(-0.6, 0.6, 50)

plt.hist(res_all, bins=bins, histtype='step', linewidth=1.5, label="All events", color="gray")
plt.hist(res_cut, bins=bins, histtype='stepfilled', alpha=0.6, label=r"$\Delta p_T < 0.20$", color="navy")

# Stats box for cut
stats_text_cut = (
    f"N = {len(res_cut)}\n"
    f"Mean = {mean_cut:.4f}\n"
    f"RMS = {rms_cut:.4f}"
    
)

plt.text(0.95, 0.95, stats_text_cut, transform=plt.gca().transAxes,
         verticalalignment='top', horizontalalignment='right',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.5), fontsize=10)


plt.title(r"$\Delta p_T < 0.3$, $p_N < 0.2$ ")
plt.xlabel(r"$(E_{\mathrm{cal}}^{\mathrm{reco}} - E_\nu^{\mathrm{true}})/E_\nu^{\mathrm{true}}$")
plt.ylabel("Events")
#plt.grid(True)
#plt.legend()
plt.tight_layout()
#plt.savefig("ecal_pT_0.2_pN_0.6.pdf")
plt.show()
